### Maintainable PySpark and Data Tests
Objectives
- Organise existing transformations into reusable functions.
- Detect invalid records and preserve rejection reasons.
- Test behaviour using explicit expected results.
- Build the unfinished franchise pipeline directly in this structure.

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F



#### 3.1 - Turning existing code into functions 

In [0]:
%sql

DESCRIBE TABLE samples.bakehouse.sales_transactions

In [0]:
%sql
-- Check whether there are any NULL values in key 
SELECT '
from samples.bakehouse.sales_transactions
WHERE transactionID IS NULL 

In [0]:
%sql

-- Check whether there are duplicate key
SELECT transactionID
FROM samples.bakehouse.sales_transactions
GROUP BY transactionID
HAVING COUNT(transactionID) > 1 

In [0]:
tables = [
    'sales_transactions',
    'sales_franchises',
    'sales_customers'
]

for t in tables :
    df = spark.read.table(f'samples.bakehouse.{t}')
    print(f'table name: {t}, row count: {df.count()}, column count: {len(df.columns)}')
    df.printSchema()
    print('-'*20)

In [0]:
# Create functions to standarize the data
def standarize_transactions(df:DataFrame) -> DataFrame :
    cols = ['transactionID', 'customerID', 'franchiseID', 'dateTime', 'product', 'quantity', 'unitPrice', 'totalPrice']
    return (
        df.select(cols)
        .withColumn('transactionID', F.col('transactionID').cast('string'))
        .withColumn('customerID', F.col('customerID').cast('string'))
        .withColumn('franchiseID', F.col('franchiseID').cast('string'))
        .withColumn('transaction_date', F.col('dateTime').cast('date'))
        .withColumn('unitPrice', F.col('unitPrice').cast('decimal(10,2)'))
        .withColumn('totalPrice', F.col('totalPrice').cast('decimal(10,2)'))
        .withColumn('calculated_total', F.col('unitPrice')*F.col('quantity').cast('decimal(10,2)'))
    )
        
def strandarize_customers(df:DataFrame) -> DataFrame :
    cols = ['customerID', 'first_name', 'last_name', 'city', 'state', 'country', 'continent', 'gender']
    return (
        df.select(cols)
        .withColumn('customerID', F.col('customerID').cast('string'))
    )

def strandarize_franchises(df:DataFrame) -> DataFrame :
    cols = ['franchiseID', 'name', 'city', 'district', 'country', 'size']
    return (
        df.select(cols)
        .withColumn('franchiseID', F.col('franchiseID').cast('string'))
        .withColumnRenamed('name', 'franchiseName')
    )

In [0]:
df_raw_transactions = spark.read.table('samples.bakehouse.sales_transactions')
df_transactions = standarize_transactions(df_raw_transactions)

df_raw_customers = spark.read.table('samples.bakehouse.sales_customers')
df_customers = strandarize_customers(df_raw_customers)

df_raw_franchises = spark.read.table('samples.bakehouse.sales_franchises')
df_franchises = strandarize_franchises(df_raw_franchises)

In [0]:
df_transactions.show(5)
df_customers.show(5)
df_franchises.show(5)

#### 3.2 - Validate sales and preserve rejects 

- missing_key : Required keys present; text keys not blank
- invalid_date : Date convertible and non-null
- invalide_product : product name non string
- invalid_quantity : Quantity convertible, non-null, and > 0
- invalid_price : Prices convertible, non-null, and ≥ 0
- price_mismatch : Absolute difference: total vs quantity * unit price <> tolerance
- unknown_customer, unknown_franchise : Customer and franchise exist in transactions tables but not in customer and franchise tables

In [0]:
def split_valid_rejected(df_transactions, df_customers, df_franchises, tolerance=0.01):
    return(
        df_transactions
        .join(
            df_customers.select('customerID').distinct().withColumn('customer_exists', F.lit(True)),
            on='customerID',
            how='left'
        )
        .join(
            df_franchises.select('franchiseID').distinct().withColumn('franchise_exists', F.lit(True)),
            on='franchiseID',
            how='left'
        )
        .withColumn(
            'dq_reason',

            # Required keys
            F.when(
                F.col('transactionID').isNull()| (F.trim(F.col('transactionID')) == '')
                | F.col('customerID').isNull() | (F.trim(F.col('customerID')) == '')
                | F.col('franchiseID').isNull() | (F.trim(F.col('franchiseID')) == ''),
                F.lit('missing_key')
            )
            # Date must be convertible and non-null
            .when(F.col('dateTime').isNull()
                | F.col('transaction_date').isNull(),
                F.lit('invalid_date')
            )
            # Product must exist and not be blank
            .when(F.col('product').isNull()
                | (F.trim(F.col('product')) == ''),
                F.lit('invalid_product')
            )
            # Quantity > 0
            .when(F.col('quantity').isNull()
                | (F.col('quantity') <= 0),
                F.lit('invalid_quantity')
            )
            # Prices >= 0
            .when(F.col('unitPrice').isNull()
                | (F.col('unitPrice') < 0)
                | F.col('totalPrice').isNull()
                | (F.col('totalPrice') < 0),
                F.lit('invalid_price')
            )
            # Difference greater than tolerance
            .when(F.abs(
                    F.col('totalPrice')
                    - F.col('calculated_total')
                ) > tolerance,
                F.lit('price_mismatch')
            )
            # Unknown customers and unknown franchises
            .when(
                F.col('customer_exists').isNull(),
                F.lit('unknown_customer')
            )
            .when(
                F.col('franchise_exists').isNull(),
                F.lit('unknown_franchise')
            )        
            .otherwise(F.lit(None))
        )
        .withColumn(
            'dq_status',
            F.when(
                F.col('dq_reason').isNull(),
                F.lit('VALID')
            ).otherwise(F.lit('INVALID'))
        )
        .withColumn(
            'checked_at',
            F.current_timestamp()
        )
    )


In [0]:
check_df = split_valid_rejected(df_transactions, df_customers, df_franchises)
check_df.show(5)

In [0]:
valid_df = check_df.filter(F.col('dq_status') == 'VALID')
invalid_df = check_df.filter(F.col('dq_status') == 'INVALID')
valid_df.show(5)
invalid_df.show(5)

In [0]:
input_rows = df_transactions.count()
output_rows = valid_df.count() + invalid_df.count()
assert input_rows == output_rows, "Input and output rows do not match"